Perform QA/QC on level 1 meteo data and store as level 2 .parquet files.

Aggregate level 2 data and store as .parquet files.

[TODO] A lot ... Generate submission-ready files.

In [ ]:
import os
# import matplotlib as plt
%matplotlib ipympl
import polars as pl

from processing.meteo import Meteo
met = Meteo()

In [ ]:
# quick'n'dirty plot of one variable
source = os.path.join(os.getcwd(), "data/level1/2022/vrxa00.parquet")
df = pl.read_parquet(source)
display(df.describe())

variable = "ta2200s0"
variable = "fkl010z0"
variable = "fa1010z0"
variable = "fkl010z1"
variable = "tre200s0"
met.plot_data(df, variable=variable)
# df_clean, errors = met.remove_extremes(df, variable=variable, q=0.001)
# met.plot_data(df_clean, variable=variable)
df.schema

In [ ]:
df.filter(pl.col(variable).is_not_null()).describe()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor

data = df.filter(pl.col(variable).is_not_null())

# Use the Sub-LOF method to detect outliers
lof = LocalOutlierFactor(n_neighbors=10, contamination='auto')
outliers = lof.fit_predict(data[variable].to_numpy().reshape((-1, 1)))
data = data.with_columns(pl.lit(outliers).alias("outliers"))
data.describe()
plt.figure(figsize=(10, 6))
plt.scatter(data["dtm"], data[variable], marker=".", s=1, label=variable)
plt.scatter(data["dtm"].filter(data["outliers"] == -1), data[variable].filter(data["outliers"] == -1), color='red', s=3, label="Outliers")
